# 08 · RAG advanced: BM25, hybrid search, reranking and contextual retrieval

**Goal:** fix the weaknesses of the semantic search from notebook 07, one technique at a time, and measure what each one changes.

| | |
|---|---|
| **You will learn** | BM25 keyword search from scratch, hybrid search with reciprocal rank fusion, reranking with Claude, and contextual retrieval with prompt caching |
| **Prerequisites** | [Notebook 07](07_rag_basics.ipynb), and a [Voyage AI](https://www.voyageai.com/) API key |
| **Run time** | About 3 minutes |
| **Cost** | Under $0.15 expected for Claude. Voyage embeddings use its free allowance. This notebook stops itself at $0.30 |
| **Paid cells** | Claude: sections 2, 3 and 4. Voyage: only for texts not already saved by notebook 07 |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

## Where we left off

Notebook 07 built semantic search and found its blind spot: **exact strings**. The incident ID `INC-2023-Q4-011` appears in three sections, but embeddings rank sections by how they *sound*, not by what they *contain*.

| Section | Technique | What it fixes |
|---|---|---|
| 1 | **BM25** keyword search | Exact strings: IDs, codes, names |
| 2 | **Hybrid search** | Combines keyword and semantic search, so each covers the other's blind spot |
| 3 | **Reranking** | A model reads each candidate with the question, and reorders them |
| 4 | **Contextual retrieval** | Gives each chunk the context it lost when it was cut out |

## Setup

The same setup as notebook 07. `track()` also counts **prompt caching** this time, which section 4 uses: on Sonnet 5.5, writing to the cache costs $2.50 per million tokens, and reading from it costs $0.10. Run the cells from top to bottom.

In [ ]:
import hashlib
import json
import math
import os
import re
from collections import Counter
from pathlib import Path

import voyageai
from anthropic import Anthropic
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

voyage_key = os.getenv("VOYAGE_API_KEY", "")
assert voyage_key and voyage_key != "your_voyage_key_here", "Add your VOYAGE_API_KEY to .env, restart the kernel, and run this again."

client = Anthropic()             # finds ANTHROPIC_API_KEY by itself
voyage_client = voyageai.Client()  # finds VOYAGE_API_KEY by itself
MODEL = "claude-sonnet-5-5"
EMBED_MODEL = "voyage-4-lite"

print("Ready. Claude model:", MODEL, "| Embedding model:", EMBED_MODEL)

In [ ]:
INPUT_PRICE = 2.00         # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00       # dollars per 1 million output tokens (Sonnet 5.5)
CACHE_WRITE_PRICE = 2.50   # dollars per 1 million tokens written to the prompt cache (5-minute cache)
CACHE_READ_PRICE = 0.10    # dollars per 1 million tokens read from the prompt cache
BUDGET_USD = 0.30          # the most this notebook should spend on Claude

spent = 0.0


def track(response):
    global spent
    usage = response.usage
    cache_write = usage.cache_creation_input_tokens or 0
    cache_read = usage.cache_read_input_tokens or 0
    cost = (usage.input_tokens * INPUT_PRICE + cache_write * CACHE_WRITE_PRICE
            + cache_read * CACHE_READ_PRICE + usage.output_tokens * OUTPUT_PRICE) / 1_000_000
    spent += cost
    cache_note = f" cache_write={cache_write:,} cache_read={cache_read:,}" if cache_write or cache_read else ""
    print(f"[input={usage.input_tokens:,}{cache_note} output={usage.output_tokens:,} "
          f"cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""

### Rebuilding notebook 07's semantic search (free when the cache exists)

This notebook stands on its own, so the next cell repeats the pieces we need from notebook 07: the section chunks, the cached embedding function, cosine similarity, the vector index and the RAG prompt. They're explained in notebook 07.

If you ran notebook 07 first, or cloned the repo with its saved embeddings, this cell makes **no Voyage calls**.

In [ ]:
report_text = Path("data/halvora_report.md").read_text(encoding="utf-8")


def chunk_by_section(text):
    return re.split(r"\n(?=## )", text)


def chunk_title(chunk):
    return chunk.splitlines()[0]


CACHE_PATH = Path("data/embeddings_cache.json")
embedding_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}


def cache_key(text, input_type):
    return hashlib.sha256(f"{EMBED_MODEL}|{input_type}|{text}".encode("utf-8")).hexdigest()


def generate_embedding(text, input_type):
    texts = [text] if isinstance(text, str) else list(text)
    missing = list(dict.fromkeys(t for t in texts if cache_key(t, input_type) not in embedding_cache))
    if missing:
        result = voyage_client.embed(missing, model=EMBED_MODEL, input_type=input_type)
        for t, vector in zip(missing, result.embeddings):
            embedding_cache[cache_key(t, input_type)] = vector
        CACHE_PATH.write_text(json.dumps(embedding_cache))
        print(f"   (Voyage call: {len(missing)} text(s), {result.total_tokens:,} tokens)")
    vectors = [embedding_cache[cache_key(t, input_type)] for t in texts]
    return vectors[0] if isinstance(text, str) else vectors


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


class VectorIndex:
    def __init__(self):
        self.vectors, self.documents = [], []

    def add_vectors(self, vectors, documents):
        self.vectors.extend(vectors)
        self.documents.extend(documents)

    def search(self, query_vector, k=3):
        scored = [(cosine_similarity(query_vector, v), d) for v, d in zip(self.vectors, self.documents)]
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return scored[:k]

    def __len__(self):
        return len(self.documents)


def build_rag_prompt(question, retrieved):
    documents = "\n\n".join(
        f'<document title="{chunk_title(document).lstrip("# ")}">\n{document}\n</document>'
        for _, document in retrieved
    )
    return f"""Answer the user's question using ONLY the documents below.
If the documents don't contain the answer, say: "The report sections I was given don't say."
Keep the answer short. End with a line "Sources:" listing the document titles you used.

<documents>
{documents}
</documents>

<user_question>
{question}
</user_question>"""


section_chunks = chunk_by_section(report_text)
index = VectorIndex()
index.add_vectors(generate_embedding(section_chunks, input_type="document"), section_chunks)

QUESTIONS = {
    "bugs": "How many bugs did engineers fix this year?",
    "revenue": "How did revenue and profit change in 2023?",
    "incident": "What happened with INC-2023-Q4-011?",
    "paraphrase": "How many defects did the developers repair?",
}
question_vectors = dict(zip(QUESTIONS, generate_embedding(list(QUESTIONS.values()), input_type="query")))

print(f"Vector index: {len(index)} section chunks | {len(QUESTIONS)} questions embedded")
assert len(index) == 16

## 1. BM25: keyword search done properly (free)

Think of a detective with a list of clues. A clue only **one** suspect has, such as a rare tattoo, is worth a lot. A clue **everyone** has, such as two eyes, is worth nothing. And finding the same clue ten times isn't ten times better than finding it once.

**BM25** ("Best Match 25") scores each chunk by the words it shares with the question, using three ideas:

| Idea | In plain words | In the formula |
|---|---|---|
| **Rare words count more** | `INC-2023-Q4-011` is in 3 of 16 chunks, so it's a strong clue | **IDF**, inverse document frequency |
| **Repeats help, but less and less** | The 10th "bug" adds much less than the 1st | **`k1`**, usually 1.2 to 2.0 |
| **Long chunks don't win by being long** | A long chunk contains more words, so its score is scaled down | **`b`**, usually 0.75 |

First, a **tokenizer** turns text into a list of lowercase words. It keeps IDs like `INC-2023-Q4-011` as one word, and drops filler words like "what" and "did".

In [ ]:
STOPWORDS = set("""a an and are as at be by did do does for from had has have how i in is it its
many much of on or our over so than that the their there these they this to was we were what
when where which who why will with you your""".split())


def tokenize(text):
    # A run of letters/digits, which may continue across "-" or "." so IDs stay whole.
    words = re.findall(r"[a-z0-9]+(?:[-.][a-z0-9]+)*", text.lower())
    return [word for word in words if word not in STOPWORDS]


for sample in [QUESTIONS["incident"], QUESTIONS["bugs"]]:
    print(f"{sample!r}\n   -> {tokenize(sample)}")

assert "inc-2023-q4-011" in tokenize(QUESTIONS["incident"])

In [ ]:
class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.documents = []
        self.doc_tokens = []        # the tokens of each document, at the same position
        self.doc_freq = Counter()   # word -> how many documents contain it

    def add_documents(self, documents):
        for document in documents:
            tokens = tokenize(document)
            self.documents.append(document)
            self.doc_tokens.append(tokens)
            self.doc_freq.update(set(tokens))   # set(): count each document once
        self.avg_len = sum(len(t) for t in self.doc_tokens) / len(self.doc_tokens)

    def idf(self, word):
        n, total = self.doc_freq[word], len(self.documents)
        return math.log((total - n + 0.5) / (n + 0.5) + 1)

    def term_scores(self, query, i):
        """How much each question word contributes to document i's score."""
        tf = Counter(self.doc_tokens[i])
        length_factor = 1 - self.b + self.b * len(self.doc_tokens[i]) / self.avg_len
        return {word: self.idf(word) * tf[word] * (self.k1 + 1) / (tf[word] + self.k1 * length_factor)
                for word in set(tokenize(query)) if word in tf}

    def search(self, query, k=3):
        """Return up to k (score, document) pairs, best first. Documents sharing no word are left out."""
        scored = [(sum(self.term_scores(query, i).values()), document) for i, document in enumerate(self.documents)]
        scored = [pair for pair in scored if pair[0] > 0]
        scored.sort(key=lambda pair: pair[0], reverse=True)
        return scored[:k]

    def __len__(self):
        return len(self.documents)


bm25_index = BM25Index()
bm25_index.add_documents(section_chunks)

print("How much a word is worth (rare means high):")
for word in ["inc-2023-q4-011", "xdr-47", "bugs", "security", "2023"]:
    print(f"   {word:<17} in {bm25_index.doc_freq[word]:>2}/{len(bm25_index)} chunks -> IDF {bm25_index.idf(word):.2f}")

**How it works**

- `add_documents` does all the counting **once**: the tokens of each chunk, how many chunks contain each word, and the average chunk length.
- `search` adds up the word scores for each chunk and **drops chunks scoring 0**. A chunk sharing no words with the question isn't a weak match: it isn't a match at all. Section 2 shows why that matters.
- BM25 takes the question as **text**, not a vector, so it needs no embedding call. It's free and instant.

**The exact-ID question from notebook 07:**

In [ ]:
question = QUESTIONS["incident"]
print(f"Question: {question}\n")
for rank, (score, document) in enumerate(bm25_index.search(question, k=5), start=1):
    has_id = "contains the ID" if "INC-2023-Q4-011" in document else ""
    print(f"   {rank}. {score:5.2f}  {chunk_title(document)[:45]:<45} {has_id}")

semantic_ranks = [r for r, (_, d) in enumerate(index.search(question_vectors["incident"], k=16), 1) if "INC-2023-Q4-011" in d]
bm25_ranks = [r for r, (_, d) in enumerate(bm25_index.search(question, k=16), 1) if "INC-2023-Q4-011" in d]
print("\nRanks of the 3 sections that contain the ID:")
print("   semantic search:", semantic_ranks)
print("   BM25           :", bm25_ranks)

assert bm25_ranks == [1, 2, 3]

All three sections containing the ID come first, because one rare, exact word is enough for BM25.

**Where BM25 fails: the same meaning in different words.** The report says engineers *fixed* 1,284 *bugs*. Ask about *developers* who *repaired defects*:

In [ ]:
paraphrase = QUESTIONS["paraphrase"]
print(f"Question: {paraphrase}")
print(f"Tokens  : {tokenize(paraphrase)}  (none of these words are in the report)\n")

print("BM25:", bm25_index.search(paraphrase, k=3) or "no results, because no words are shared")
print("\nSemantic search:")
for score, document in index.search(question_vectors["paraphrase"], k=3):
    print(f"   {score:.3f}  {chunk_title(document)}")

assert bm25_index.search(paraphrase) == []

Each method fails where the other is strong:

| | Exact IDs, codes, names | Same meaning, different words |
|---|---|---|
| **Semantic (embeddings)** | Weak | Strong |
| **Keyword (BM25)** | Strong | Blind |

That table is the reason for the next section.

## 2. Hybrid search: combining both rankings (paid at the end)

Two judges rank the same 16 contestants. One judges meaning and the other exact words. Their **scores** use different scales, such as 0.34 against 2.38, so you can't just add them. But you can combine their **rankings**: a contestant placed near the top by both judges wins.

**Reciprocal rank fusion (RRF)** gives each document $\frac{1}{60 + \text{rank}}$ from each list it appears in, and adds them up.

- It uses **ranks, not scores**, so different scales don't matter and nothing needs tuning.
- **The 60** comes from the original RRF paper. It stops rank 1 from completely dominating rank 2.

In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    """Merge lists of (score, document), each best first. Returns (rrf_score, document), best first."""
    fused = {}
    for ranked in ranked_lists:
        for rank, (_, document) in enumerate(ranked, start=1):
            fused[document] = fused.get(document, 0) + 1 / (k + rank)
    return sorted(((score, doc) for doc, score in fused.items()), key=lambda pair: pair[0], reverse=True)


# A toy example with letters. The scores in the input are ignored: only the order counts.
semantic_toy = [(0.9, "A"), (0.8, "B"), (0.7, "C")]
keyword_toy = [(5.0, "C"), (3.0, "A")]   # B shares no words, so it's missing here
for score, document in reciprocal_rank_fusion([semantic_toy, keyword_toy]):
    print(f"   {document}: {score:.4f}")

**A** wins: it's near the top of **both** lists. **C** comes close, last in one list but first in the other. **B** appears in only one list, so it falls behind.

`Retriever` runs both searches, fuses them, and can show where each chunk came from.

In [ ]:
class Retriever:
    def __init__(self, vector_index, bm25_index):
        self.vector_index = vector_index
        self.bm25_index = bm25_index

    def search(self, question, k=3, explain=False):
        query_vector = generate_embedding(question, input_type="query")
        semantic = self.vector_index.search(query_vector, k=len(self.vector_index))   # full rankings
        keyword = self.bm25_index.search(question, k=len(self.bm25_index))
        fused = reciprocal_rank_fusion([semantic, keyword])

        if explain:
            semantic_rank = {doc: r for r, (_, doc) in enumerate(semantic, 1)}
            keyword_rank = {doc: r for r, (_, doc) in enumerate(keyword, 1)}
            print(f"   {'hybrid':>6} {'semantic':>9} {'BM25':>5}   section")
            for r, (_, doc) in enumerate(fused[:k], 1):
                print(f"   {r:>6} {semantic_rank[doc]:>9} {keyword_rank.get(doc, '-'):>5}   {chunk_title(doc)[:50]}")
        return fused[:k]


retriever = Retriever(index, bm25_index)

print(f"Q: {QUESTIONS['incident']}")
incident_top = retriever.search(QUESTIONS["incident"], k=3, explain=True)
print(f"\nQ: {paraphrase}")
retriever.search(paraphrase, k=3, explain=True)

assert sum("INC-2023-Q4-011" in doc for _, doc in incident_top) == 3

**What to notice**

- **The incident question:** all three ID sections are in the hybrid top 3, whatever semantic search thought of them.
- **The paraphrase question:** BM25 has no results, shown as `-`, so the hybrid ranking simply follows semantic search. This is why BM25 drops zero scores. If it had returned 16 tied zeros as a "ranking", RRF would have mixed that noise in.
- The `explain` table is how you debug retrieval: for any chunk, you can see which method found it.

**All four methods, the same questions, one table:**

In [ ]:
def word_match_search(question, k=3):
    """Plain word counting: no rare-word weighting and no length correction."""
    words = tokenize(question)
    scored = [(sum(tokenize(doc).count(w) for w in words), doc) for doc in section_chunks]
    scored = [pair for pair in scored if pair[0] > 0]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:k]


methods = {
    "word match": lambda q: word_match_search(q, k=3),
    "semantic": lambda q: index.search(generate_embedding(q, input_type="query"), k=3),
    "BM25": lambda q: bm25_index.search(q, k=3),
    "hybrid": lambda q: retriever.search(q, k=3),
}
tests = [
    (QUESTIONS["bugs"], lambda docs: any("Section 2:" in chunk_title(d) for d in docs)),
    (QUESTIONS["incident"], lambda docs: sum("INC-2023-Q4-011" in d for d in docs) == 3),
    (paraphrase, lambda docs: any("Section 2:" in chunk_title(d) for d in docs)),
    (QUESTIONS["revenue"], lambda docs: any("Section 3:" in chunk_title(d) for d in docs)),
]

print(f"{'question':<46}" + "".join(f"{name:>12}" for name in methods))
for question, passed in tests:
    row = "".join(f"{'pass' if passed([d for _, d in search(question)]) else 'FAIL':>12}" for search in methods.values())
    print(f"{question[:44]:<46}{row}")

print("\nGoals: Section 2 in the top 3 | all 3 ID sections in the top 3 | Section 2 in the top 3 | Section 3 in the top 3")

**How to read it:** word match and BM25 should fail the paraphrase row, because no words are shared. Semantic search may fail the ID row. **Hybrid** should be the only column that passes every row. It isn't magic: it takes the best of both lists.

With 16 chunks and 4 questions this is a demonstration, not a benchmark. Real systems test retrieval on many questions, in the same way notebook 03 tested prompts.

**The RAG flow from notebook 07, now with hybrid retrieval.** Only the retrieval step changes. The prompt is reused unchanged.

In [ ]:
def rag_answer_hybrid(question, k=3):
    print("STEP 1-2: hybrid search (semantic + BM25, fused with RRF):")
    retrieved = retriever.search(question, k=k, explain=True)
    response = track(client.messages.create(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": build_rag_prompt(question, retrieved)}],
    ))
    print("\nSTEP 3: Claude answers\n")
    print(text_from_message(response))
    return response


hybrid_response = rag_answer_hybrid(QUESTIONS["incident"])
assert text_from_message(hybrid_response)

**Retrieved isn't the same as used.** Claude may leave Section 12 out of its answer, because the question asks what *happened*, and Section 12 is about future plans. That's correct behaviour. Retrieval's job is to make the right chunks **available**. Deciding what's relevant is Claude's job.

## 3. Reranking: a second, smarter look (paid)

A librarian pulls 8 books from the catalogue in seconds: that's search. Then an expert skims each one **with your question in mind** and hands you the best 3: that's reranking. The catalogue is fast but rough, and the expert is slow but careful.

Search alone has a limit. Embeddings turn each chunk into a vector **before** anyone asks a question, so the chunk is never read *together with* the question. A **reranker** reads the question and every candidate together, and judges which ones actually answer it.

**The usual pattern:** retrieve **wide** (the top 8, cheaply), rerank, and keep the best **3** for the prompt.

> **Different from the course:** the course gets the reranker's answer as JSON with assistant prefill, which current models reject. We use structured outputs from notebook 02, so the answer comes back as a validated list of numbers.

In [ ]:
class Ranking(BaseModel):
    document_ids: list[int]


def rerank_with_claude(question, candidates, k=3):
    """candidates: (score, document) pairs from any search. Returns the best k, reordered."""
    documents = "\n\n".join(f'<document id="{i}">\n{doc}\n</document>' for i, (_, doc) in enumerate(candidates))
    prompt = f"""You are ranking search results for a question.

<user_question>
{question}
</user_question>

<documents>
{documents}
</documents>

Pick the {k} documents that are MOST useful for answering the question, most useful first.
Return their ids."""
    response = track(client.messages.parse(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": prompt}],
        output_format=Ranking,
    ))
    ids = response.parsed_output.document_ids
    ids = [i for i in dict.fromkeys(ids) if 0 <= i < len(candidates)]   # drop duplicates and bad ids
    print(f"   Claude ranked ids {ids}")
    return [candidates[i] for i in ids][:k]


question = QUESTIONS["revenue"]
candidates = retriever.search(question, k=8)   # retrieve wide

print(f"Q: {question}\n\nHybrid search, top 8 before reranking:")
for rank, (_, document) in enumerate(candidates, start=1):
    print(f"   {rank}. {chunk_title(document)}")

print("\nAfter reranking, keeping 3:")
reranked = rerank_with_claude(question, candidates, k=3)
for rank, (_, document) in enumerate(reranked, start=1):
    print(f"   {rank}. {chunk_title(document)}")

assert 1 <= len(reranked) <= 3

**What to notice**

- Each candidate gets a number, and Claude returns only the numbers, not the text. That keeps the answer tiny and fast.
- **We still check the ids.** Duplicates and out-of-range numbers are dropped. Never trust model output to be perfectly formed, even with structured outputs.
- Reranking often matters most by **pushing out chunks that look relevant but aren't**, such as a methodology section that only *talks about* checking financial figures. The prompt then holds three useful chunks instead of two plus noise.

**Hybrid against hybrid + rerank** on the four test questions. This makes four reranking calls.

In [ ]:
rerank_tests = [
    (QUESTIONS["bugs"], "Section 2 in top 3", lambda docs: any("Section 2:" in chunk_title(d) for d in docs)),
    (QUESTIONS["incident"], "all 3 ID sections in top 3", lambda docs: sum("INC-2023-Q4-011" in d for d in docs) == 3),
    (paraphrase, "Section 2 in top 3", lambda docs: any("Section 2:" in chunk_title(d) for d in docs)),
    (QUESTIONS["revenue"], "Section 3 in top 2", lambda docs: any("Section 3:" in chunk_title(d) for d in docs[:2])),
]

rows = []
for question, goal, passed in rerank_tests:
    print(f"Q: {question}")
    wide = retriever.search(question, k=8)
    hybrid_top3 = [d for _, d in wide[:3]]
    reranked_top3 = [d for _, d in rerank_with_claude(question, wide, k=3)]
    rows.append((question, goal, passed(hybrid_top3), passed(reranked_top3)))

print(f"\n{'question':<46}{'goal':<30}{'hybrid':>8}{'+rerank':>9}")
for question, goal, hybrid_ok, rerank_ok in rows:
    print(f"{question[:44]:<46}{goal:<30}{'pass' if hybrid_ok else 'FAIL':>8}{'pass' if rerank_ok else 'FAIL':>9}")

**What to notice**

- Watch the **incident** row. A reranker judges **usefulness**, not "contains the ID". For *"What happened with..."* it may drop Section 12, which only says what the company will do next. That's a reasonable choice for answering, but wrong if you need **every** chunk that mentions something, such as for an audit. Use exact search like BM25 for that.
- Reranking costs a Claude call per question, adding time and tokens. It's worth it when search returns many almost-right chunks, and often unnecessary for 16 clean, well-titled sections.

## 4. Contextual retrieval: give every chunk its missing context (paid)

Think of a photocopied page from the middle of a book, with no chapter title. *"It rose from 61% to 74%"*: **what** rose? Contextual retrieval writes a **sticky note** on each page before filing it: *"From the Software Engineering section, about the share of bugs found before release."*

This is Anthropic's **contextual retrieval** technique:

1. For each chunk, send Claude **the whole document plus the chunk**, and ask for 1 or 2 sentences that place the chunk in the document.
2. Put that context in front of the chunk.
3. Embed **and** BM25-index the contextualised chunk instead of the bare one.

We use the 500-character chunks from notebook 07 here, not the section chunks. Section chunks already start with their `## ` title, so they barely need help, but size-based chunks often start mid-sentence with no header.

In [ ]:
def chunk_by_char(text, chunk_size=500, chunk_overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start = end - chunk_overlap if end < len(text) else len(text)
    return chunks


char_chunks = chunk_by_char(report_text)
with_header = sum("## " in chunk for chunk in char_chunks)
print(f"{len(char_chunks)} size-based chunks, and only {with_header} contain a '## ' header.\n")

for i in [10, 29]:
    print(f"--- char_chunks[{i}], first 220 characters ---")
    print(char_chunks[i][:220].replace("\n", " ") + " ...\n")

Read those chunks cold: which division? Which year? Which incident? A person can't tell, and neither can an embedding model or BM25. The facts are there, but the **words that connect them to a question** are in a different chunk.

### Prompt caching makes this affordable

Step 1 sends the **same whole report** with every chunk: 30 times. That's exactly what **prompt caching** is for. We mark the report with `"cache_control": {"type": "ephemeral"}`, and put it **first**, before the part that changes:

- **The first call writes** the report to the cache, at 1.25 times the normal input price.
- **Later calls read** it from the cache, at a twentieth of the normal price on Sonnet 5.5, for 5 minutes after each use.
- A prompt shorter than **512 tokens** isn't cached on Sonnet 5.5, and the API doesn't report an error, so always check the usage numbers. Our report is several thousand tokens, so it qualifies.

> **Different from `claude-api-101`:** there, on Haiku 4.5, this report was too short to cache, because that model's minimum is 4,096 tokens. Sonnet 5.5's minimum is 512, so caching works here.

The results are also saved to `data/contextual_chunks_cache.json`, so a second run, or a clone of the repo, costs nothing.

In [ ]:
CONTEXT_CACHE_PATH = Path("data/contextual_chunks_cache.json")
context_cache = json.loads(CONTEXT_CACHE_PATH.read_text()) if CONTEXT_CACHE_PATH.exists() else {}


def add_context(chunk, document):
    key = hashlib.sha256(f"{MODEL}|{chunk}".encode("utf-8")).hexdigest()
    if key in context_cache:
        return context_cache[key]

    response = track(client.messages.create(
        model=MODEL,
        max_tokens=300,
        output_config={"effort": "low"},
        messages=[{"role": "user", "content": [
            # Block 1: the whole document. It's identical every time, so it's cached.
            {"type": "text", "text": f"<document>\n{document}\n</document>",
             "cache_control": {"type": "ephemeral"}},
            # Block 2: the chunk and the instruction. This part changes on every call.
            {"type": "text", "text": f"""Here is a chunk from the document above:
<chunk>
{chunk}
</chunk>

Write 1-2 short sentences that situate this chunk within the overall document, to improve
search retrieval of the chunk. Mention the section it comes from and what it is about.
Answer only with the context sentences, nothing else."""},
        ]}],
    ))
    context = text_from_message(response).strip()
    context_cache[key] = context
    CONTEXT_CACHE_PATH.write_text(json.dumps(context_cache, indent=2))
    return context


def contextualize(chunk, document):
    """The text we actually index: the context first, then the original chunk, unchanged."""
    return f"{add_context(chunk, document)}\n\n{chunk}"


contextual_chunks = [contextualize(chunk, report_text) for chunk in char_chunks]

print(f"\nContextualised {len(contextual_chunks)} chunks.\n")
for i in [10, 29]:
    print(f"--- contextual_chunks[{i}], first 360 characters ---")
    print(contextual_chunks[i][:360].replace("\n", " ") + " ...\n")

assert len(contextual_chunks) == len(char_chunks)

**What to notice**

- **Read the cost lines from the top.** The first call should show `cache_write`, and the calls after it `cache_read`, each costing much less. If every call shows `cache_write`, the cache expired or something before the report changed.
- If no cost lines appeared at all, every context came from the saved file and cost nothing.
- **Context can be wrong.** A chunk that straddles two sections can confuse the context step too. That's one more reason to cut on structure when you can, and to read a few generated contexts yourself.

**Plain chunks against contextual chunks.** Same 30 chunks, same hybrid search, same questions. The only difference is the context in front of each chunk. Each question has a phrase that only the right chunk contains, and we report that chunk's rank.

In [ ]:
plain_vectors = generate_embedding(char_chunks, input_type="document")
contextual_vectors = generate_embedding(contextual_chunks, input_type="document")


def build_retriever(chunks, vectors):
    vector_index, bm25 = VectorIndex(), BM25Index()
    vector_index.add_vectors(vectors, chunks)
    bm25.add_documents(chunks)
    return Retriever(vector_index, bm25)


plain_retriever = build_retriever(char_chunks, plain_vectors)
contextual_retriever = build_retriever(contextual_chunks, contextual_vectors)

context_tests = [
    ("What share of bugs did engineers find before release?", "61% to 74%"),
    ("How much of its revenue does the company spend on marketing?", "4.4% of revenue"),
    ("What security plans does the company have for 2024?", "keep investing in security"),
    ("Why did some research partnerships end?", "Two partnerships ended"),
    ("How did the engineers respond to the security incident?", "Engineers patched the library"),
]
generate_embedding([q for q, _ in context_tests], input_type="query")   # one batched call for all questions


def rank_of(retriever_, question, phrase):
    ranked = retriever_.search(question, k=len(retriever_.vector_index))
    return next(r for r, (_, doc) in enumerate(ranked, start=1) if phrase in doc)


print(f"\n{'question':<62}{'plain':>7}{'contextual':>12}")
for question, phrase in context_tests:
    plain, contextual = rank_of(plain_retriever, question, phrase), rank_of(contextual_retriever, question, phrase)
    change = "better" if contextual < plain else ("same" if contextual == plain else "worse")
    print(f"{question[:60]:<62}{plain:>7}{contextual:>12}   {change}")
print(f"\nRank of the chunk containing the answer, out of {len(char_chunks)}. 1 is best.")

**What to notice**

- Rows that were already **#1** were never the problem: their chunks already contain the question's key words. Contextual retrieval mostly **rescues hard chunks**, the ones whose connecting words were somewhere else.
- A wrong context sentence rarely makes things much worse, because the original words are still in the chunk. It just fails to help.
- Anthropic's published experiments found that contextual retrieval cut retrieval failures substantially, and more when combined with BM25 and reranking: the full stack this notebook built.

## Final spend

In [ ]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget on Claude.")
print(f"Saved embeddings: {len(embedding_cache)} | saved contexts: {len(context_cache)}")

## The whole RAG pipeline (notebooks 07 and 08)

| Step | Notebook | What we built |
|---|---|---|
| Cut the document | 07 | `chunk_by_char`, `chunk_by_sentence`, `chunk_by_section` |
| Add context (optional) | 08 | `add_context`, `contextualize`, with prompt caching |
| Turn chunks into vectors | 07 | `generate_embedding`, batched and cached |
| Index | 07, 08 | `VectorIndex`, `BM25Index` |
| Search | 08 | `Retriever`: semantic + BM25, fused with RRF |
| Rerank (optional) | 08 | `rerank_with_claude` |
| Answer | 07 | `build_rag_prompt` + Claude, answering only from the chunks, with sources |

## Takeaways

1. **BM25 is keyword search with three fixes:** rare words count more, repeats count less and less, and long chunks are scaled down.
2. **Tokenizing matters.** Keep IDs whole and drop filler words, or BM25 scores the wrong things.
3. **Each search has a blind spot:** semantic search misses exact strings, and BM25 misses paraphrases.
4. **Hybrid search with RRF combines ranks, not scores,** so nothing needs tuning. Only fuse real matches.
5. **A reranker reads the question and each candidate together.** Retrieve wide, rerank, keep a few. It optimises for answering, not for finding every mention.
6. **Contextual retrieval puts 1 or 2 sentences of context in front of each chunk.** It costs one call per chunk, once, and prompt caching makes those calls cheap.
7. **RAG is modular.** We changed retrieval three times without touching the answer step.

## Your turn

The first two exercises are free. The others cost a few cents.

- In section 1, set `STOPWORDS = set()` and re-run. Which chunk jumps up for the incident question, and why?
- In section 2, change `k=60` to `k=1` in the toy RRF example. How does a small `k` change the result?
- In section 3, change the incident question to *"Which sections mention INC-2023-Q4-011?"* Does the reranker keep Section 12 now?
- In section 4, delete the `cache_control` line and the file `data/contextual_chunks_cache.json`, then re-run section 4. Compare its total with your first run: the difference is what prompt caching saved. This costs about 25 cents, so raise `BUDGET_USD` to `0.60` first, or the budget stop will end the run partway.

**Next:** [09 · Claude features](09_claude_features.ipynb): extended thinking, images, PDFs, citations and prompt caching.